# Monte Carlo: DML-DiD when sparsity fails
`dmldid.py` estimator · `dgp.py` designs · `mc.py` replication, diagnostics, tables

In [ ]:
# imports
import numpy as np
import pandas as pd
import dgp, mc

pd.set_option('display.width', 200)
R = 500  # replications (use ~20 to test)

gauss  = dict(rep=mc.replicate, sim=dgp.sim_gauss,  bases=dgp.bases_gauss)
cat    = dict(rep=mc.replicate, sim=dgp.sim_cat,    bases=dgp.bases_cat)
center = dict(rep=mc.replicate, sim=dgp.sim_center, bases=dgp.bases_center)

## D1: Chang's DGP, then with a trend
(a) $\ell_{10}=1$, sd .1, $N=200$, $p=100$. (b) Same $\gamma$, trend $1+X'\gamma$ with $R^2=.5$, $p=N/2$.

In [2]:
gam = lambda p: np.r_[np.arange(5, 0, -1) / 5, np.zeros(p - 5)]  # Chang's code
d1 = pd.concat(
    [mc.run(**gauss, R=R, N=200, K=K, gamma=gam(100), delta=np.zeros(100), sigma=.1, sigma3=.1)
       .assign(design='chang', K=K, N=200) for K in (2, 5)] +
    [mc.run(**gauss, R=R, N=N, gamma=gam(N // 2), delta=gam(N // 2), sigma=np.sqrt(2.2), sigma3=.1)
       .assign(design='trend', K=5, N=N) for N in (200, 1000, 2000)])
mc.summary(d1, by=('design', 'K', 'N', 'est')).round(3)

n   bias     sd   rmse  cover  se_sd  bias_sd     B1  err_g  err_l  nsel_g  nsel_l  kappa   c1   c2   c3
design K N    est                                                                                                                   
chang  2 200  bcch        500 -0.000  0.025  0.025  0.952  0.962   -0.018 -0.000  0.249  0.011   0.533   0.000  0.147  0.0  0.0  0.0
              oracle      500  0.001  0.024  0.024  0.958  0.974    0.023    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
              ordid        24  0.008  0.057  0.056  0.958  1.155    0.145    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
              unadjusted  500 -0.000  0.017  0.017  0.950  1.013   -0.002    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
       5 200  bcch        500 -0.002  0.030  0.030  0.948  0.864   -0.067 -0.000  0.203  0.009   1.290   0.004  0.050  0.0  0.0  0.0
              oracle      500  0.001  0.025  0.025  0.956  0.982    0.031    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
              ordid        24  0.008  0.057  0.056  0.958  1.155    0.145    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
              unadjusted  500 -0.000  0.017  0.017  0.950  1.013   -0.002    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
trend  5 200  bcch        500  0.538  0.782  0.949  0.598  0.747    0.688  0.565  0.203  1.600   1.290   0.179  0.050  0.0  0.0  0.0
              oracle      500  0.023  0.346  0.346  0.956  0.949    0.066    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
              ordid        24  0.090  0.852  0.839  1.000  1.124    0.106    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
              unadjusted  500  1.571  0.283  1.597  0.000  0.973    5.550    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
         1000 bcch        500  0.093  0.191  0.212  0.886  0.908    0.487  0.087  0.077  0.492   3.359   3.080  0.009  0.0  0.0  0.0
              oracle      500  0.006  0.159  0.159  0.962  0.992    0.038    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
              unadjusted  500  1.567  0.122  1.572  0.000  1.012   12.870    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
         2000 bcch        500  0.038  0.117  0.123  0.932  0.972    0.323  0.033  0.048  0.247   3.963   3.978  0.005  0.0  0.0  0.0
              oracle      500  0.001  0.111  0.111  0.952  0.986    0.005    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
              unadjusted  500  1.560  0.090  1.563  0.000  0.970   17.369    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN

## D2–D3: density path and alignment
Decay $\gamma_j \propto j^{-a}$, unit norm, trend $R^2=.5$. D2: $\cos(\gamma,\delta)=1$. D3: $\cos\in\{1,.5,0\}$.

In [3]:
def density(Ns, As, cos=(1.,), ratio=.5):
    out = []
    for N in Ns:
        for a in As:
            for c in cos:
                g, d = dgp.coefs(int(ratio * N), 'decay', a, c)
                out.append(mc.run(**gauss, R=R, N=N, gamma=g, delta=d)
                             .assign(N=N, a=a, cos=c, naive=dgp.naive_bias(g, d)))
    return pd.concat(out)

d2 = density((250, 500, 1000, 2000), (2., 1., .75, .5))
s2 = mc.summary(d2, by=('est', 'a', 'cos', 'N'))
display(s2.round(3), mc.slopes(s2.loc[['bcch']]).round(2))

n   bias     sd   rmse  cover  se_sd  bias_sd  naive     B1  err_g  err_l  nsel_g  nsel_l  kappa   c1   c2   c3
est        a    cos N                                                                                                                      
bcch       0.50 1.0 250   500  0.791  0.183  0.811  0.014  1.007    4.332  0.826  0.801  0.213  1.091   0.122   0.054  0.362  0.0  0.0  0.0
                    500   500  0.753  0.128  0.764  0.002  0.993    5.870  0.826  0.782  0.210  1.083   0.293   0.262  0.319  0.0  0.0  0.0
                    1000  500  0.707  0.093  0.713  0.000  0.945    7.567  0.826  0.698  0.200  1.019   0.819   0.886  0.234  0.0  0.0  0.0
                    2000  500  0.669  0.067  0.672  0.000  0.914    9.926  0.826  0.651  0.191  0.962   1.743   1.976  0.186  0.0  0.0  0.0
           0.75 1.0 250   500  0.550  0.243  0.601  0.238  0.861    2.266  0.826  0.578  0.192  1.000   0.533   0.354  0.191  0.0  0.0  0.0
                    500   500  0.477  0.133  0.496  0.044  0.980    3.588  0.826  0.491  0.170  0.850   1.087   1.112  0.121  0.0  0.0  0.0
                    1000  500  0.396  0.094  0.407  0.010  0.936    4.212  0.826  0.388  0.154  0.733   1.744   2.074  0.090  0.0  0.0  0.0
                    2000  500  0.317  0.069  0.325  0.006  0.874    4.584  0.826  0.316  0.138  0.635   2.871   3.725  0.065  0.0  0.0  0.0
           1.00 1.0 250   500  0.316  0.252  0.404  0.614  0.848    1.254  0.826  0.345  0.154  0.791   0.861   0.724  0.102  0.0  0.0  0.0
                    500   500  0.287  0.130  0.315  0.394  0.982    2.211  0.826  0.298  0.133  0.641   1.200   1.285  0.076  0.0  0.0  0.0
                    1000  500  0.202  0.095  0.223  0.354  0.904    2.119  0.826  0.198  0.116  0.504   1.737   2.244  0.057  0.0  0.0  0.0
                    2000  500  0.139  0.061  0.152  0.350  0.965    2.276  0.826  0.139  0.098  0.412   2.588   3.546  0.041  0.0  0.0  0.0
           2.00 1.0 250   500  0.057  0.187  0.196  0.934  1.017    0.307  0.826  0.068  0.074  0.345   0.996   0.970  0.053  0.0  0.0  0.0
                    500   500  0.065  0.118  0.135  0.906  1.026    0.546  0.826  0.075  0.067  0.301   1.010   1.036  0.046  0.0  0.0  0.0
                    1000  500  0.045  0.088  0.099  0.902  0.950    0.506  0.826  0.049  0.062  0.248   1.054   1.353  0.038  0.0  0.0  0.0
                    2000  500  0.019  0.058  0.061  0.944  1.010    0.330  0.826  0.018  0.053  0.155   1.293   1.962  0.030  0.0  0.0  0.0
oracle     0.50 1.0 250   500 -0.011  0.160  0.161  0.948  1.011   -0.067  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                    500   500 -0.005  0.114  0.114  0.956  1.014   -0.047  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                    1000  500  0.005  0.080  0.080  0.946  1.011    0.061  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                    2000  500  0.000  0.061  0.061  0.936  0.943    0.004  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
           0.75 1.0 250   500 -0.013  0.171  0.172  0.932  0.942   -0.077  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                    500   500  0.003  0.112  0.112  0.952  1.030    0.028  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                    1000  500  0.004  0.080  0.080  0.944  1.014    0.047  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                    2000  500 -0.001  0.061  0.061  0.936  0.937   -0.019  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
           1.00 1.0 250   500 -0.015  0.169  0.169  0.960  0.966   -0.089  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                    500   500  0.001  0.114  0.114  0.962  1.012    0.008  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                    1000  500 -0.000  0.082  0.082  0.942  0.987   -0.000  0.826    NaN    NaN    NaN     Na

est   a     cos
bcch  0.50  1.0    0.40
      0.75  1.0    0.33
      1.00  1.0    0.25
      2.00  1.0    0.02
dtype: float64

In [4]:
d3 = density((500, 1000), (1., .5), cos=(1., .5, 0.))
mc.summary(d3, by=('est', 'a', 'cos', 'N')).round(3)

n   bias     sd   rmse  cover  se_sd  bias_sd  naive     B1  err_g  err_l  nsel_g  nsel_l  kappa   c1   c2   c3
est        a   cos N                                                                                                                      
bcch       0.5 0.0 500   500  0.044  0.133  0.140  0.946  0.995    0.332 -0.000  0.042  0.210  0.990   0.293   0.386  0.319  0.0  0.0  0.0
                   1000  500  0.059  0.098  0.114  0.876  0.931    0.599 -0.000  0.054  0.200  0.940   0.819   1.325  0.234  0.0  0.0  0.0
               0.5 500   500  0.379  0.127  0.400  0.158  1.025    2.984  0.413  0.387  0.210  0.991   0.293   0.421  0.319  0.0  0.0  0.0
                   1000  500  0.348  0.094  0.361  0.036  0.952    3.701  0.413  0.343  0.200  0.933   0.819   1.300  0.234  0.0  0.0  0.0
               1.0 500   500  0.753  0.128  0.764  0.002  0.993    5.870  0.826  0.782  0.210  1.083   0.293   0.262  0.319  0.0  0.0  0.0
                   1000  500  0.707  0.093  0.713  0.000  0.945    7.567  0.826  0.698  0.200  1.019   0.819   0.886  0.234  0.0  0.0  0.0
           1.0 0.0 500   500  0.035  0.130  0.135  0.952  1.017    0.269  0.000  0.028  0.133  0.607   1.200   1.538  0.076  0.0  0.0  0.0
                   1000  500  0.007  0.086  0.086  0.958  0.997    0.077 -0.000  0.011  0.116  0.467   1.737   3.023  0.057  0.0  0.0  0.0
               0.5 500   500  0.135  0.126  0.185  0.816  0.985    1.074  0.413  0.135  0.133  0.554   1.200   1.058  0.076  0.0  0.0  0.0
                   1000  500  0.089  0.084  0.122  0.824  1.007    1.058  0.413  0.090  0.116  0.445   1.737   2.250  0.057  0.0  0.0  0.0
               1.0 500   500  0.287  0.130  0.315  0.394  0.982    2.211  0.826  0.298  0.133  0.641   1.200   1.285  0.076  0.0  0.0  0.0
                   1000  500  0.202  0.095  0.223  0.354  0.904    2.119  0.826  0.198  0.116  0.504   1.737   2.244  0.057  0.0  0.0  0.0
oracle     0.5 0.0 500   500 -0.005  0.114  0.114  0.956  1.014   -0.047 -0.000    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                   1000  500  0.005  0.080  0.080  0.946  1.011    0.061 -0.000    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
               0.5 500   500 -0.005  0.114  0.114  0.956  1.014   -0.047  0.413    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                   1000  500  0.005  0.080  0.080  0.946  1.011    0.061  0.413    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
               1.0 500   500 -0.005  0.114  0.114  0.956  1.014   -0.047  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                   1000  500  0.005  0.080  0.080  0.946  1.011    0.061  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
           1.0 0.0 500   500  0.001  0.114  0.114  0.962  1.012    0.008  0.000    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                   1000  500 -0.000  0.082  0.082  0.942  0.987   -0.000 -0.000    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
               0.5 500   500  0.001  0.114  0.114  0.962  1.012    0.008  0.413    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                   1000  500 -0.000  0.082  0.082  0.942  0.987   -0.000  0.413    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
               1.0 500   500  0.001  0.114  0.114  0.962  1.012    0.008  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                   1000  500 -0.000  0.082  0.082  0.942  0.987   -0.000  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
ordid      0.5 0.0 500     4 -0.223  0.228  0.298  1.000  1.493   -0.977 -0.000    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
               0.5 500     4 -0.223  0.228  0.298  1.000  1.493   -0.977  0.413    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
               1.0 500     4 -0.223  0.228  0.298  1.000  1.493   -0.977  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  

## D4: which nuisance is dense
Sparse core (5 coords) or 80% of the norm in an equal-weight tail.

In [5]:
N, p = 1000, 500
d4 = pd.concat([mc.run(**gauss, R=R, N=N, gamma=g, delta=d).assign(dense_g=i, dense_l=j, naive=dgp.naive_bias(g, d))
                for i in (False, True) for j in (False, True)
                for g, d in [(dgp.core_tail(p, dense=i), dgp.core_tail(p, dense=j))]])
mc.summary(d4, by=('dense_g', 'dense_l', 'est')).round(3)

n   bias     sd   rmse  cover  se_sd  bias_sd  naive     B1  err_g  err_l  nsel_g  nsel_l  kappa   c1   c2   c3
dense_g dense_l est                                                                                                                          
False   False   bcch        500  0.001  0.090  0.090  0.944  0.958    0.011  0.826 -0.002  0.079  0.190   4.230   4.792  0.038  0.0  0.0  0.0
                oracle      500  0.001  0.080  0.080  0.954  1.014    0.013  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                unadjusted  500  0.829  0.085  0.833  0.000  1.003    9.718  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
        True    bcch        500  0.028  0.118  0.121  0.946  1.056    0.233  0.370  0.024  0.079  1.018   4.230   0.144  0.038  0.0  0.0  0.0
                oracle      500  0.001  0.080  0.080  0.954  1.014    0.013  0.370    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                unadjusted  500  0.371  0.088  0.381  0.012  1.009    4.217  0.370    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
True    False   bcch        500  0.005  0.063  0.063  0.958  1.033    0.086  0.370  0.002  0.209  0.130   0.139   4.967  0.395  0.0  0.0  0.0
                oracle      500  0.003  0.079  0.079  0.968  1.032    0.038  0.370    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                unadjusted  500  0.373  0.086  0.383  0.010  1.032    4.343  0.370    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
        True    bcch        500  0.816  0.084  0.821  0.000  1.032    9.685  0.826  0.810  0.209  1.087   0.139   0.135  0.395  0.0  0.0  0.0
                oracle      500  0.003  0.079  0.079  0.968  1.032    0.038  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN
                unadjusted  500  0.829  0.084  0.833  0.000  1.023    9.913  0.826    NaN    NaN    NaN     NaN     NaN    NaN  NaN  NaN  NaN

## D5: reference category
$J$ fixed (bounded inflation) vs. $J=N/20$ (inflation growing with $p$); `hit` = which nuisance depends on the categoricals. `full` = all $J$ dummies (remedy, excluded from $R$).

In [ ]:
d5 = pd.concat([mc.run(**cat, R=R, N=N, J=J, hit=h).assign(N=N, J=J, rule=rule, hit=h)
                for N in (500, 1000, 2000) for rule, J in (('fixed', 5), ('growing', N // 20))
                for h in ('g', 'l', 'both')])
display(mc.summary(d5, by=('rule', 'hit', 'N', 'est', 'basis')).round(3),
        mc.ranges(d5, by=('rule', 'hit', 'N')).round(2))

## D6: overlap
Scale the propensity index; $|c_1|$ and $R$ against realized $\hat\kappa$.

In [ ]:
d6 = pd.concat([mc.run(**cat, R=R, N=1000, J=10, scale=s).assign(scale=s) for s in (.5, 1., 2., 3.)])
display(mc.summary(d6[d6.basis.isin(['A0', 'bad'])], by=('scale', 'basis')).round(3),
        mc.ranges(d6, by=('scale',)).round(2))

## D7: centring before the cubic expansion
Bounded sparsity inflation. Cubic fits extrapolate at extreme test points, so read B1 with care.

In [ ]:
d7 = pd.concat([mc.run(**center, R=R, N=N).assign(N=N) for N in (500, 1000, 2000, 4000)])
display(mc.summary(d7, by=('N', 'est', 'basis')).round(3), mc.ranges(d7, by=('N',)).round(2))

## D8: repeated cross sections on Sequeira's covariates
Chang's sample ($N=1084$) from `Table 1/Bribes_Regression.dta` in his replication repo. Real $D$ and $X$; $T$ and $Y$ simulated with known $\theta_0$. `post_2008` is $T$, so it is left out of $X$.

In [ ]:
keep = ['lba', 'tariff_change_2008', 'post_2008', 'tariff2007', 'lvalue_tonnage', 'differentiated', 'agri',
        'perishable', 'dfs', 'day_w_arrival', 'monitor', 'psi', 'rsa', 'clear_agent', 'hc_group', 'hc_4digits']
seq = pd.read_stata('Bribes_Regression.dta', convert_categoricals=False)[keep].dropna()  # N = 1084
X = seq[keep[3:11]].to_numpy(float)
D = seq.tariff_change_2008.to_numpy().astype(int)

S = dgp.resolve(dgp.poly(X, X.mean(0)))  # drop degenerate dummy powers
Qs = {k: dgp.poly(X, c)[:, S] for k, c in (('A0', X.mean(0)), ('raw', 0.), ('median', np.median(X, 0)))}
cg, cl = dgp.sparse_coefs(Qs['A0'], seed=1), dgp.sparse_coefs(Qs['A0'], seed=2)
d8 = mc.run(mc.replicate_rcs, R=R, Qs=Qs, cg=cg, cl=cl, D=D, lam=seq.post_2008.mean())
display(mc.summary(d8, by=('est', 'basis')).round(3), mc.ranges(d8).round(2))